# CAMS CO: эксперимент SAM3D V2 с глобальным декодером

Запускайте ячейки последовательно из репозитория. Сначала распаковка и просмотр, затем длительное обучение. Исходные архивы сохраняются; понадобится место под распакованные NetCDF и два массива выбранного региона.

Гипотеза текущего запуска: SAM3D CAE превосходит PCA при одинаковой размерности 64. Модель обучается один раз на полном пуле 2020–2021, checkpoint выбирается по 2022, итоговая оценка выполняется на 2023. Результат заранее не предполагается.


In [ ]:
# При необходимости выполните один раз и перезапустите kernel:
# %pip install numpy pandas matplotlib netCDF4 scipy scikit-learn torch
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/models.py').exists()), None)
assert ROOT is not None, 'Откройте ноутбук внутри репозитория DeepCompreesion'
sys.path.insert(0,str(ROOT))
from scripts.cams_experiment import (unpack, inspect_files, experiment_directory, prepare,
                                     split_days, run_experiments)
from src.models import get_model

CONFIG = {
    'data_dir': 'data/cams_co_cross_year',
    'variable': None,  # автоопределение co / co_conc; при необходимости задайте имя
    'selected_levels': [0, 250, 500, 1000, 2000, 5000], # эксперимент на шести высотах
    'center_lat_lon': [51.0, 10.0],  # географический центр домена CAMS Europe
    'crop_shape': [96, 84],         # реальные ячейки, без интерполяции
    'train_day_counts': [], # один запуск на всём train pool (2020–2021)
    'split_years': {'train': [2020, 2021], 'validation': 2022, 'test': 2023},
    'evaluation_days_per_month': 7, # из каждого скачанного месяца val/test
    'gap_days': 1,
    'subset_seed': 42,
    'latent_dims': [64],
    'methods': ['PCA', 'SAM3DAutoencoderV2'], # сначала PCA, затем новая SAM3D V2
    'evaluation_splits': ['validation', 'test'], # train-метрики не считаются
    'seeds': [42],                 # пилот; для оценки разброса задайте [42, 43, 44] ДО запуска
    'epochs': 100,                 # все эпохи; сохраняется минимум validation MSE
    'batch_size': 16,
    'learning_rate': 0.001,
    'weight_decay': 0.000001,
    'dropout': 0.0,                # для точной реконструкции без train/eval-сдвига
    'mae_weight': 0.0,             # PCA и нейросеть оптимизируются по одному MSE
}


## 1. Распаковать и посмотреть содержимое
Архивы извлекаются в `data/cams_co/unpacked/`. Повторный запуск пропускает завершённую распаковку. Число кадров в таблице относится к конкретному файлу: если высоты лежат в разных файлах, складывать эти числа нельзя. Следующая ячейка собирает уникальные timestamps и высоты и выявляет дубли/пропуски.


In [ ]:
files = unpack(ROOT/CONFIG['data_dir'], CONFIG['selected_levels'])
inventory, records, reference = inspect_files(files, CONFIG['variable'], CONFIG['selected_levels'])
display(inventory)
print('Исходная сетка (lat, lon):',len(reference[0]),len(reference[1]))
print('Единицы CO:',reference[2],'; единицы высоты:',reference[3])
print('Всего уникальных timestamps:',len({t for r in records for t in r['dates']}))
print('Высоты:',sorted({float(z) for r in records for z in r['levels']}))


## 2. Подготовить регион и показать поля
На диск сохраняются массивы `(time, level, latitude, longitude)` в float32. В память читается один пространственный кадр выбранного региона, не весь месячный файл.

Нормализация совпадает по смыслу с предыдущим экспериментом: min–max для каждого кадра и высоты. Это обратимое преобразование, использующее дополнительные `2 × число высот` чисел на кадр; учитываем их в таблице. Физические метрики вычисляются после обратного преобразования в единицах исходного файла. Метрики CAMS нельзя напрямую смешивать с WRF-Chem.


In [ ]:
selected_files = sorted({record['path'] for record in records})
OUTPUT = experiment_directory(ROOT,CONFIG,selected_files)
inventory.to_csv(OUTPUT/'inventory.csv',index=False)
raw, X, minima, scales, frames, summary = prepare(records,reference,CONFIG,OUTPUT)
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Результаты:',OUTPUT)
fig, axes = plt.subplots(1,3,figsize=(15,4))
levels_to_show = [0,len(summary['levels'])//2,len(summary['levels'])-1]
for ax,z in zip(axes,levels_to_show):
    im=ax.imshow(raw[0,z],origin='upper',aspect='auto')
    ax.set_title(f"Первый кадр, высота {summary['levels'][z]} {summary['level_units']}")
    ax.set_xlabel('Индекс longitude'); ax.set_ylabel('Индекс latitude')
    fig.colorbar(im,ax=ax,label=summary['units'])
fig.tight_layout(); fig.savefig(OUTPUT/'first_frame.png',dpi=150); plt.show()
plt.figure(figsize=(12,3))
plt.plot(frames.time,np.mean(raw,axis=(1,2,3)))
plt.ylabel(f"Среднее CO, {summary['units']}"); plt.xlabel('Время'); plt.tight_layout()
plt.savefig(OUTPUT/'co_time_series.png',dpi=150); plt.show()

# Архитектура создаётся и выводится сразу после подготовки данных.
neural_method = next(name for name in CONFIG['methods'] if name != 'PCA')
model = get_model(neural_method, latent_dim=CONFIG['latent_dims'][0],
                  input_shape=(1, *X.shape[1:]), dropout_rate=CONFIG['dropout'])
print(f'Архитектура перед обучением: {neural_method}')
print(model)
print(f'Вход: {(1, *X.shape[1:])}; latent_dim={CONFIG["latent_dims"][0]}; '
      f'параметров={sum(p.numel() for p in model.parameters()):,}')
del model


## 3. Зафиксировать разбиение
Полные 2020–2021 годы образуют train pool. Из каждого скачанного месяца 2022 выбираются 7 равномерно расположенных дней для validation, а из каждого скачанного месяца 2023 — 7 дней для test. Test нельзя использовать для выбора архитектуры или эпохи.

Пространственные патчи из одного timestamp не размножаются между выборками. Соседние дни всё равно могут коррелировать; это пилотное временное разбиение, не независимые атмосферные реализации. PCA fit выполняется только на train.


In [ ]:
splits, counts = split_days(frames,CONFIG,OUTPUT)
display(pd.read_csv(OUTPUT/'partitions.csv').groupby('partition').agg(frames=('frame','size'),days=('day','nunique'),first=('time','min'),last=('time','max')))
display(pd.DataFrame([{'train_days':n,'frames':len(splits[f'train_{n}'])} for n in counts]))
assert min(len(splits[f'train_{n}']) for n in counts) >= max(CONFIG['latent_dims']), 'Увеличьте минимальный train или уменьшите latent_dims для PCA'
print('Число комбинаций:',len(counts)*len(CONFIG['latent_dims'])*len(CONFIG['seeds'])*len(CONFIG['methods']))


## 4. Обучение и оценка (долгая ячейка)
Notebook запускает ровно два метода: сначала PCA, затем `SAM3DAutoencoderV2`. В V2 из одного latent-вектора декодер строит глобальную линейную реконструкцию и уточняет её локальной SAM-свёрточной ветвью; размер передаваемого кода остаётся равным `latent_dim`. Транспонированные свёртки точно обращают три пространственных downsampling-шага с помощью `output_padding`, финальная интерполяция не используется. Сеть проходит все 100 эпох, сохраняется минимум validation MSE. Полные reconstruction-метрики считаются только на validation и test; train используется для fit без долгого покадрового scoring.

PCA обучается только на train. `payload_bytes` показывает размер представления одного кадра без упаковки; отдельно указаны min/max-нормировка и число параметров AE. Размер обученных весов и PCA-базиса не входит в payload.


In [ ]:
metrics = run_experiments(X,raw,minima,scales,frames,splits,counts,CONFIG,OUTPUT)
display(metrics[metrics.split=='test'][['method','latent_dim','train_days','seed','mse','rmse','relative_l2','ssim','physical_rmse','best_epoch']])


## 5. Итоговое сравнение на фиксированном test
Главный результат — сравнение `SAM3DAutoencoderV2` и PCA на фиксированном test 2023 после обучения на полном пуле 2020–2021. Validation используется для выбора checkpoint, а test — только для итоговых метрик.

CSV содержит pooled MSE/MAE/RMSE, среднюю покадровую relative L2 и средний SSIM по срезам (data_range=1), физические ошибки и покадровые/подневные результаты. При одном seed нет оценки устойчивости к инициализации; стандартное отклонение по трём seed также не является доверительным интервалом по независимым датасетам.


In [ ]:
test = metrics[metrics.split=='test']
columns=['mse','rmse','mae','relative_l2','ssim','physical_rmse']
summary_table=test.groupby(['method','latent_dim','train_days'])[columns].agg(['mean','std','count'])
summary_table.to_csv(OUTPUT/'test_summary.csv'); display(summary_table)
for dim in CONFIG['latent_dims']:
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,metric in zip(axes,['mse','relative_l2','ssim']):
        for name,group in test[test.latent_dim==dim].groupby('method'):
            stat=group.groupby('train_days')[metric].agg(['mean','std']).sort_index()
            ax.plot(stat.index,stat['mean'],marker='o',label=name)
            if stat['std'].notna().any():
                ax.fill_between(stat.index,stat['mean']-stat['std'],stat['mean']+stat['std'],alpha=.15)
        ax.set_xlabel('Обучающих дней'); ax.set_ylabel(metric); ax.grid(alpha=.2)
    axes[0].legend(fontsize=7); fig.suptitle(f'CAMS, latent dimension = {dim}')
    fig.tight_layout(); fig.savefig(OUTPUT/f'learning_curve_dim{dim}.png',dpi=180); plt.show()
print('Для анализа пришлите metrics_all_runs.csv, data_summary.json и partitions.csv из',OUTPUT)
